# Moving Morphable Components (MMC) Journey

This notebook tells the MMC side of the story: domain discretization, optimization loops, and load-case studies.



## Story Outline

1. **Imports & MMC Modules**  
   - Bring in numpy/pandas/plotly/scipy, and import `mmc_core` functions.  
   - Config dataclasses for default L-bracket domain + loadcases.
2. **Geometry & Mesh Setup**  
   - Visualize finite element grid vs void mask using Plotly heatmaps.  
   - Show how component radius/beta map to physical features.
3. **Optimization Loop Walkthrough**  
   - Step-by-step explanation of `run_mmc_lbracket`, including design variable updates.  
   - Include small interactive widget to adjust radius/lr and view predicted step.
4. **Horizontal Load Case Results**  
   - Rebuild `mmc_compliance_lbracket.png`, `mmc_paths_lbracket.png` inline using `mmc_lbracket_log.csv`.  
   - Provide 3D scatter of screw trajectories extruded in pseudo-time for clarity.
5. **Vertical Load Case & Generalization**  
   - Mirror plots using `mmc_log_vertical.csv`.  
   - Compare compliance envelopes vs horizontal case.
6. **Performance & Comparisons**  
   - Compute wall-clock metrics, normalized compliance, and pass data back to master notebook.
7. **Reproduction**  
   - Document how to run `run_mmc_lbracket.py` with CLI flags, including environment requirements.



In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

_possible_dirs = [Path.cwd().resolve()]
if (Path.cwd() / 'docs' / 'notebooks').exists():
    _possible_dirs.append(Path.cwd() / 'docs' / 'notebooks')
helper_dir = None
for candidate in _possible_dirs:
    candidate = candidate.resolve()
    if (candidate / 'story_helpers.py').exists():
        helper_dir = candidate
        break
if helper_dir is None:
    raise RuntimeError('Cannot locate story_helpers.py')
if str(helper_dir) not in sys.path:
    sys.path.append(str(helper_dir))

from story_helpers import (
    ROOT,
    RESULTS_DIR,
    draw_lbracket_2d,
    make_bracket_mesh,
)

# Configure Plotly for VS Code Jupyter extension
try:
    pio.renderers.default = "vscode"
except ValueError:
    try:
        pio.renderers.default = "notebook"
    except ValueError:
        # Fallback: let Plotly auto-detect
        pass

MMC_SRC_DIR = ROOT / 'src' / 'approach_b_mmc'
if str(MMC_SRC_DIR) not in sys.path:
    sys.path.append(str(MMC_SRC_DIR))

from mmc_core import (
    MMCConfig,
    DomainConfig,
    ConstraintConfig,
    run_mmc_lbracket,
    update_design,
    build_passive_mask,
)


## Geometry & Finite-Element Grid

We reuse the shared L-bracket visualizations and extend them with MMC-specific overlays (void mask, passive regions).



In [ ]:
draw_lbracket_2d().show()
make_bracket_mesh().show()

def plot_passive_mask(domain: DomainConfig):
    mask = build_passive_mask(domain)
    fig = px.imshow(mask, origin="lower", color_continuous_scale=["#f5f5f5", "#37474f"],
                    title="Passive Region Mask (1 = void)")
    fig.update_xaxes(title="X element")
    fig.update_yaxes(title="Y element")
    return fig

plot_passive_mask(DomainConfig()).show()



## MMC Optimization Logs

We load the canonical MMC runs (`mmc_lbracket_log.csv`, `mmc_log_vertical.csv`), reconstruct compliance curves, and visualize screw trajectories in both 2D and pseudo-3D.



In [ ]:
def load_mmc_log(tag: str) -> pd.DataFrame:
    path = RESULTS_DIR / f"mmc_{tag}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

horizontal_log = load_mmc_log("lbracket_log")
vertical_log = load_mmc_log("log_vertical")

comp_fig = go.Figure()
comp_fig.add_trace(
    go.Scatter(x=horizontal_log["iter"], y=horizontal_log["compliance"], mode="lines+markers", name="Horizontal")
)
comp_fig.add_trace(
    go.Scatter(x=vertical_log["iter"], y=vertical_log["compliance"], mode="lines+markers", name="Vertical")
)
comp_fig.update_layout(title="MMC Compliance vs Iteration", xaxis_title="Iteration", yaxis_title="Compliance (F^T u)")
comp_fig.show()

def plot_trajectory(df: pd.DataFrame, title: str) -> go.Figure:
    fig = go.Figure()
    for idx in [1, 2]:
        fig.add_trace(
            go.Scatter(
                x=df["x" + str(idx)],
                y=df["y" + str(idx)],
                mode="lines+markers",
                name=f"Screw {idx}",
                text=[f"iter {i}" for i in df["iter"]],
            )
        )
    fig.update_layout(title=title, xaxis_title="X (elements)", yaxis_title="Y (elements)")
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig

plot_trajectory(horizontal_log, "Horizontal Load Trajectory").show()
plot_trajectory(vertical_log, "Vertical Load Trajectory").show()

traj3d = go.Figure()
for df, label, color in [
    (horizontal_log, "Horizontal", "#1f77b4"),
    (vertical_log, "Vertical", "#ff7f0e"),
]:
    traj3d.add_trace(
        go.Scatter3d(
            x=df["x1"],
            y=df["y1"],
            z=df["iter"],
            mode="lines",
            name=f"Screw 1 {label}",
            line=dict(color=color),
        )
    )
traj3d.update_layout(title="3D Trajectory (iteration as Z)", scene=dict(zaxis_title="Iteration"))
traj3d.show()



In [ ]:
def summarize_log(df: pd.DataFrame, label: str) -> dict:
    return {
        "label": label,
        "iterations": df["iter"].max(),
        "final_compliance": df["compliance"].iloc[-1],
        "avg_wall_time_ms": df.get("wall_time_ms", pd.Series([0])).mean(),
    }

summary_df = pd.DataFrame([
    summarize_log(horizontal_log, "Horizontal"),
    summarize_log(vertical_log, "Vertical"),
])
display(summary_df)



## Interactive MMC Sandbox

Experiment with a reduced-iteration MMC run by tuning `radius` and `lr`. This keeps runtime short (<=10 iterations) while showing how the component field responds.



In [ ]:
def sandbox(radius=4.0, lr=0.5):
    cfg = MMCConfig(n_iters=10, lr=lr, radius=radius)
    history = run_mmc_lbracket(cfg)
    df = pd.DataFrame(history)
    display(df.tail())
    plot_trajectory(df, f"Sandbox Trajectory (radius={radius}, lr={lr})").show()

widgets.interact(
    sandbox,
    radius=widgets.FloatSlider(value=4.0, min=2.0, max=8.0, step=0.5),
    lr=widgets.FloatLogSlider(value=0.5, base=10, min=-2, max=0.5),
);



## Reproduction Notes

```
cd $REPO_ROOT
python src/approach_b_mmc/run_mmc_lbracket.py --iters 30 --load-case horizontal_tip --tag lbracket
python src/approach_b_mmc/run_mmc_lbracket.py --iters 30 --load-case vertical_tip --tag vertical
```

The notebook automatically reloads the resulting CSVs and refreshes the figures above. Update the `tag` argument to keep multiple experiments on disk.



## Linking Back to the Story

- Feed `sandbox` outputs into `master_story.ipynb` for unified comparisons.  
- Use `pinn_story.ipynb`'s `pinn_predict` helper to contrast MMC placements with surrogate predictions.  
- Re-run this notebook whenever new MMC configurations are tested.

